<a href="https://colab.research.google.com/github/curiousbrutus/fNIRS-Vise/blob/main/notebooks/run_on_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# fMRI-fNIRS Transfer Learning on Google Colab

This notebook runs the fMRI-teacher → fNIRS-student transfer learning pipeline on Google Colab with GPU acceleration.

**Key Features:**
- Optimized for Colab K80/T4 GPUs (12-16GB VRAM)
- Automatic data download from OSF
- Real-time training monitoring with W&B
- Memory-efficient training with FP16

**Runtime Requirements:**
- GPU runtime (K80, T4, or better)
- ~2GB disk space for model and data
- ~1 hour training time for 100 epochs

## 🚀 Setup & Installation

In [ ]:
# Clone the repository
!git clone https://github.com/curiousbrutus/fNIRS-Vise.git
%cd fNIRS-Vise

# Verify we're in the right directory
!ls -la

In [ ]:
# Install the package and dependencies
%pip install -e .

# Verify GPU is available
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 📊 W&B Logging Setup (Optional)

Set up Weights & Biases for experiment tracking. You can skip this if you prefer not to log.

In [ ]:
# Optional: Login to W&B for experiment tracking
import wandb

# Uncomment and run if you want W&B logging
# wandb.login()

# Or set to offline mode
import os
os.environ["WANDB_MODE"] = "offline"  # Comment this line if you want online logging

print("W&B setup complete (offline mode)")

## 🧠 Train the Transfer Learning Model

Run the training with Colab-optimized configuration.

In [ ]:
# Train with Colab configuration
!python -m src.train \
    --config configs/colab.yaml \
    --fnirs_data_path ./data/fNIRS/processed \
    --fmri_data_path ./data/fMRI/features \
    --batch_size 8 \
    --max_epochs 100 \
    --accelerator gpu \
    --devices 1 \
    --precision 16 \
    --transfer_mode feature_guided \
    --freeze_fmri \
    --learning_rate 3e-4 \
    --experiment_name "colab_transfer_learning"

### Alternative: Quick Training (Reduced Epochs)

For faster experimentation, run with fewer epochs:

In [ ]:
# Quick training (20 epochs)
!python -m src.train \
    --batch_size 8 \
    --max_epochs 20 \
    --accelerator gpu \
    --devices 1 \
    --precision 16 \
    --transfer_mode feature_guided \
    --experiment_name "colab_quick_test"

## 📈 Monitor Training Progress

In [ ]:
# Check GPU memory usage
from src.utils.memory_guard import MemoryGuard

MemoryGuard.log_memory_stats()

# List training checkpoints
!find . -name "*.ckpt" -type f | head -10

## 📊 Model Evaluation

Evaluate the trained model and visualize results.

In [ ]:
# Find the best checkpoint
import glob
checkpoints = glob.glob("./lightning_logs/*/checkpoints/*.ckpt")

if checkpoints:
    best_checkpoint = sorted(checkpoints)[-1]  # Get latest
    print(f"Using checkpoint: {best_checkpoint}")
    
    # Run evaluation
    !python scripts/evaluate.py \
        --mode single \
        --checkpoint_path "{best_checkpoint}" \
        --test_subject sub01
else:
    print("No checkpoints found. Train a model first.")

## 🔬 Interactive Demo

Test the model with live predictions.

In [ ]:
# Run the demo script
!python demo.py

# Interactive prediction example
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.models.fmri_fnirs_net import FmriFnirsNet

# Create a demo model
model = FmriFnirsNet()

# Generate synthetic fNIRS and fMRI data
fnirs_demo = torch.randn(1, 52, 200) * 0.1  # Realistic fNIRS scale
fmri_demo = torch.randn(1, 768) * 0.5       # Realistic fMRI scale

# Make prediction
with torch.no_grad():
    logits = model(fnirs_demo, fmri_demo)
    probabilities = torch.softmax(logits, dim=-1)
    prediction = torch.argmax(logits, dim=-1)

# Visualize results
emotion_labels = ["Neutral", "Happy", "Sad", "Angry"]

plt.figure(figsize=(12, 4))

# Plot fNIRS signal (first few channels)
plt.subplot(1, 3, 1)
plt.plot(fnirs_demo[0, :5, :].T)  # Plot first 5 channels
plt.title("fNIRS Signals (5 channels)")
plt.xlabel("Time")
plt.ylabel("Amplitude")

# Plot fMRI features (first 20 dimensions)
plt.subplot(1, 3, 2)
plt.bar(range(20), fmri_demo[0, :20])
plt.title("fMRI Features (first 20)")
plt.xlabel("Feature Index")
plt.ylabel("Value")

# Plot prediction probabilities
plt.subplot(1, 3, 3)
plt.bar(emotion_labels, probabilities[0].numpy())
plt.title(f"Prediction: {emotion_labels[prediction[0]]}")
plt.ylabel("Probability")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print(f"\nPredicted emotion: {emotion_labels[prediction[0]]} (confidence: {probabilities[0].max():.3f})")

## 🔄 Hyperparameter Sweep (Optional)

Run a small hyperparameter sweep to find optimal settings.

In [ ]:
# Run a limited hyperparameter sweep
transfer_modes = ["feature_guided", "distill"]
distill_alphas = [0.3, 0.5, 0.7]

results = []

for mode in transfer_modes:
    if mode == "distill":
        for alpha in distill_alphas:
            print(f"\n=== Training {mode} with alpha={alpha} ===")
            
            !python -m src.train \
                --transfer_mode {mode} \
                --distill_alpha {alpha} \
                --batch_size 8 \
                --max_epochs 10 \
                --accelerator gpu \
                --precision 16 \
                --experiment_name "colab_sweep_{mode}_a{alpha}"
    else:
        print(f"\n=== Training {mode} ===")
        
        !python -m src.train \
            --transfer_mode {mode} \
            --batch_size 8 \
            --max_epochs 10 \
            --accelerator gpu \
            --precision 16 \
            --experiment_name "colab_sweep_{mode}"

print("\n✅ Hyperparameter sweep completed!")

## 💾 Download Results

Download trained models and results to your local machine.

In [ ]:
# Create a zip file with results
import shutil
from google.colab import files

# Package results
!zip -r colab_results.zip lightning_logs/ checkpoints/ wandb/ -x "*/wandb/run-*/files/*" "*/__pycache__/*"

# Download the results
files.download('colab_results.zip')

print("📦 Results packaged and ready for download!")
print("The zip contains:")
print("  - Training logs and metrics")
print("  - Model checkpoints")
print("  - W&B experiment data")

## 🧹 Cleanup (Optional)

Clean up temporary files to free up space.

In [ ]:
# Clean up large temporary files
!rm -rf ./data/*/  # Remove downloaded data
!rm -rf wandb/*/files/  # Remove large W&B files

# Show remaining disk usage
!du -sh * | sort -hr | head -10

print("🧹 Cleanup completed!")

## 🎉 Conclusion

You've successfully run the fMRI-fNIRS transfer learning pipeline on Google Colab!

**What you accomplished:**
- ✅ Trained a brain signal transfer learning model
- ✅ Optimized for GPU memory constraints
- ✅ Monitored training with real-time metrics
- ✅ Evaluated model performance
- ✅ Generated interactive predictions

**Next steps:**
1. **Use your own data**: Replace dummy data with real fNIRS/fMRI recordings
2. **Optimize hyperparameters**: Run full hyperparameter sweeps
3. **Deploy the model**: Use the trained model for real-time brain decoding
4. **Scale up**: Train on multiple GPUs or TPUs for larger datasets

**Resources:**
- [GitHub Repository](https://github.com/curiousbrutus/fNIRS-Vise)
- [Transfer Learning Documentation](https://github.com/curiousbrutus/fNIRS-Vise/blob/main/README_TRANSFER.md)
- [Paper: fNIRS-Vise](https://github.com/curiousbrutus/fNIRS-Vise/blob/main/Paper_fNIRS-Vise.pdf)

**Contact**: eyyub.gvn@gmail.com for questions and collaborations! 🧠✨